# data_normalization.ipynb

---

This block is used to convert data from ```generate-data.py``` to the format for ```sde_model.ipynb```.

While the implementation could have been done at the data generation level, this acts as an interoperable bridge betwen Edgar Fernández's code and Lucas Berredo's.

---

This block is used to convert data from ```generate-data.py``` to the format for ```route_optimizer.py```.

While the implementation could have been done at the data generation level, this acts as an interoperable bridge betwen Edgar Vigil's code and Juan Boudón's.

In [ ]:
import pandas as pd
import numpy as np
import torch
from scipy.interpolate import interp1d
import os
import random

# Rutas
csv_path = "/home/edgar/GitHub/StochasticATM/dataset_listo_para_SDE.csv"
output_dir = "../Datasets/"
output_path = os.path.join(output_dir, "flight_data.pt")
test_output_path = os.path.join(output_dir, "flight_data_test.pt")

def preprocess_and_save_flights(csv_path, output_path, test_output_path, num_steps=200):
    print("Cargando dataset...")
    df = pd.read_csv(csv_path)
    
    # 1. Eliminar nulos de la base
    df = df.dropna(subset=['time', 'vel_mps', 'baro_altitude', 'fuel_flow_kgs', 'track', 'vrate_mps'])
    
    # 2. Separar Trayectorias por Saltos de Tiempo (> 1200s = 20 minutos)
    df['flight_id'] = df['icao24'].astype(str) + "_" + df['callsign'].astype(str)
    df = df.sort_values(by=['flight_id', 'time'])
    df['time_gap'] = df.groupby('flight_id')['time'].diff() > 1200
    df['unique_trajectory_id'] = df['flight_id'] + "_" + df.groupby('flight_id')['time_gap'].cumsum().astype(str)

    # 3. Arreglar el Track (Seno y Coseno)
    df['track_sin'] = np.sin(np.radians(df['track']))
    df['track_cos'] = np.cos(np.radians(df['track']))
    
    # =====================================================================
    # 4. CAPA DE DEFENSA 1: LÍMITES FÍSICOS ESTRICTOS (CLIPPING)
    # =====================================================================
    # Recortamos todo lo que sea físicamente imposible para un avión comercial
    print("Aplicando límites físicos estrictos...")
    df['vrate_mps'] = df['vrate_mps'].clip(lower=-25.0, upper=25.0)
    df['vel_mps'] = df['vel_mps'].clip(lower=0.0, upper=340.0) # ~Mach 1
    df['baro_altitude'] = df['baro_altitude'].clip(lower=0.0, upper=14000.0) # ~45,000 ft
    
    # Para el combustible, evitamos los negativos y cortamos el 0.1% de picos más extremos
    fuel_max_logical = df['fuel_flow_kgs'].quantile(0.999) 
    df['fuel_flow_kgs'] = df['fuel_flow_kgs'].clip(lower=0.0, upper=fuel_max_logical)

    smooth_features = ['fuel_flow_kgs', 'vel_mps', 'baro_altitude', 'vrate_mps']

    # =====================================================================
    # 5. CAPA DE DEFENSA 2: EL DESTRUCTOR DE PICOS (FILTRO DE MEDIANA)
    # =====================================================================
    # La mediana (a diferencia de la media) ignora por completo los valores 
    # atípicos. Si el radar salta 1 segundo a 100%, la mediana lo borra mágicamente.
    print("Destruyendo ruido de radar (Filtro de Mediana)...")
    for feat in smooth_features:
        df[feat] = df.groupby('unique_trajectory_id')[feat].transform(
            lambda x: x.rolling(window=7, min_periods=1, center=True).median()
        )

    # =====================================================================
    # 5.5. CAPA DE DEFENSA 3: EL SUAVIZADO FINAL (FILTRO DE MEDIA)
    # =====================================================================
    # Ahora que los "fantasmas" han sido borrados, aplicamos la media para que 
    # la trayectoria sea una curva suave y fluida (perfecta para SDEs)
    print("Suavizando señales para crear trayectorias fluidas...")
    for feat in smooth_features:
        df[feat] = df.groupby('unique_trajectory_id')[feat].transform(
            lambda x: x.rolling(window=15, min_periods=1, center=True).mean()
        )

    # 6. Variables Finales para el Modelo (6 dimensiones)
    features = ['fuel_flow_kgs', 'vel_mps', 'baro_altitude', 'vrate_mps', 'track_sin', 'track_cos']

    # 7. Normalización Global Min-Max
    stats = {}
    for feat in features:
        if feat in ['track_sin', 'track_cos']:
            f_min, f_max = -1.0, 1.0
        else:
            f_min, f_max = df[feat].min(), df[feat].max()
            
        denom = (f_max - f_min) if f_max > f_min else 1.0
        df[feat] = (df[feat] - f_min) / denom
        stats[feat] = {'min': float(f_min), 'max': float(f_max)}
    
    # 8. Creación de Tensores Interpolados
    unique_trajectories = df['unique_trajectory_id'].unique()
    all_flights_data = []
    
    print(f"Procesando {len(unique_trajectories)} trayectorias únicas coherentes...")
    for tid in unique_trajectories:
        traj_df = df[df['unique_trajectory_id'] == tid].copy()
        
        # Ignorar trayectorias demasiado cortas
        if len(traj_df) < 50:
            continue
            
        t_orig = traj_df['time'].values
        t_norm = (t_orig - t_orig.min()) / (t_orig.max() - t_orig.min())
        t_new = np.linspace(0, 1, num_steps)
        
        resampled_features = []
        for feat in features:
            f_interp = interp1d(t_norm, traj_df[feat].values, kind='linear', bounds_error=False, fill_value="extrapolate")
            resampled_features.append(f_interp(t_new))
        
        all_flights_data.append(np.stack(resampled_features, axis=-1))
    
    if len(all_flights_data) < 2:
        print("Error: No hay suficientes datos limpios para separar en Train y Test.")
        return

    # Mezclar la lista de vuelos aleatoriamente
    random.shuffle(all_flights_data)

    # Extraer uno al azar
    test_flight_data = all_flights_data.pop()
    
    train_tensor = torch.tensor(np.stack(all_flights_data), dtype=torch.float32) 
    test_tensor = torch.tensor(test_flight_data, dtype=torch.float32).unsqueeze(0) 

    y_true_train = train_tensor.permute(1, 0, 2)
    y_true_test = test_tensor.permute(1, 0, 2)

    t_grid = torch.linspace(0, 1, num_steps)
    
    # 10. Guardar Archivos
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    torch.save({'y_true': y_true_train, 't': t_grid, 'stats': stats}, output_path)
    torch.save({'y_true': y_true_test, 't': t_grid, 'stats': stats}, test_output_path)
    
    print("-" * 40)
    print(f"ÉXITO: Preprocesamiento Completado")
    print(f"Features (Dim = {len(features)}): {features}")
    print(f"TRAIN Shape: {list(y_true_train.shape)} [Time, Batch, Features]")
    print(f"TEST Shape:  {list(y_true_test.shape)} [Time, Batch, Features]")
    print("-" * 40)

if __name__ == "__main__":
    preprocess_and_save_flights(csv_path, output_path, test_output_path)

In [ ]:
import pandas as pd
import numpy as np

imported_df = pd.read_csv("../Datasets/dataset_trayectorias_completas.csv")

# We need to create a tensor, serialized, with columns "Fuel", "Velocity", "Altitude", "Weather"

FileNotFoundError: [Errno 2] No such file or directory: '../Datasets/dataset_trayectorias_completas.csv'